In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [2]:
import pandas as pd
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm
import catboost as cb

from models.utilities import dw
from data_helpers.preprocessors.scalers import denormalize_feats, normalize_feats, normalize_other

In [3]:
model_name = 'gbt'

## Prepare and Filter Dataset

In [4]:
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft').reset_index(drop=True)
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

cat_feats = ['feedback_setting', 'price_rule']
time_aggregated_dataset = pd.get_dummies(time_aggregated_dataset, columns=cat_feats)
dummy_vars_columns = list(filter(lambda x: 'feedback_setting_' in x or 'price_rule_' in x ,time_aggregated_dataset.columns))

## Load and Filter Dataset

In [5]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
feature_cols = [col for col in time_aggregated_dataset.columns.values if "change" not in col and "running_" in col]
time_columns =  ['n_unique_deals_round', 'round']# + ['round']
rounds = range(1,5)
n_deal_prices = range(0,6)
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)

## Set model hyper-param grid for search with CV

In [6]:
min_quantile = 0.35
max_quantile = 1 - min_quantile
hyperparameter_grid = dict(n_estimators=[10, 20, 30],
                                        depth=[5, 6,8],
                                        learning_rate=[0.001, 0.01, 0.1],  
                                        min_data_in_leaf=[1,2,5]
                                       )

## Fit and evaluate models

In [7]:
np.random.seed(1)
all_results = []
regression_res = []

for i in tqdm(range(pdl.max_samples)):
    train_df, test_df = pdl.get_sample_split_dataset(i)
    # Perform a new cross validation for each round
    sub_train_df = train_df
    gbt_regressor = cb.CatBoostRegressor(boost_from_average=True, 
                                     loss_function='Quantile',
                                     grow_policy='SymmetricTree',
                                     task_type="CPU",
                                    )
    # Normalize values with median and IQR.
    X = sub_train_df[feature_cols].values
    X_norm, X_median, X_iqr = normalize_feats(X, min_quantile, max_quantile)
    y = sub_train_df['allocative_efficiency_round'].values[:, np.newaxis]
    
    # Add extra input features from price rule, feedback settign and time related features.
    X_other = sub_train_df[dummy_vars_columns+time_columns].values
    X_all = np.concatenate([X_norm, X_other], axis=1)
    
    # perfrom a grid search to find the best set of hyper-parameters.
    best_model_params = gbt_regressor.grid_search(hyperparameter_grid,
                                         X_all,
                                         y=y,
                                         cv=5,
                                         partition_random_seed=0,
                                         calc_cv_statistics=False,
                                         search_by_train_test_split=True,
                                         refit=True, # refit the best model on the whole dataset.
                                         shuffle=False,
                                         stratified=None,
                                         train_size=0.5,
                                         verbose=False,
                                         plot=False,
                                         log_cout=dw,
                                         log_cerr=sys.stderr,
                                  )
    
    # get the best model, that is refitted on the whole dataset.
    best_model = gbt_regressor    
    # get feature importance.
    assert len(feature_cols + dummy_vars_columns+time_columns) == len(best_model.feature_importances_)
    feat_importance_df = pd.Series(dict(zip(
    feature_cols + dummy_vars_columns+time_columns, best_model.feature_importances_.tolist()))).to_frame().T
    feat_importance_df['sample_id'] = i

    # get the results of the best parametrization after the grid search.
    best_params_df = pd.Series(best_model_params['params']).to_frame().T
    a = pd.concat([feat_importance_df, best_params_df], axis=1, ignore_index=False)
    regression_res.append(a)

    for rd in rounds:  

        
        for n_deal_price in n_deal_prices:
            # fetch the respective test set.
            test_query = 'round == ' + str(rd) + ' and n_unique_deals_round ==  ' + str(n_deal_price)
            sub_test_set = test_df.query(test_query).copy()
            
            # normalize price feats of test set.
            X_test_price_feats = sub_test_set[feature_cols].values
            X_test_norm, X_test_median, X_test_iqr = normalize_feats(X_test_price_feats, min_quantile, max_quantile)
            
            # add additional feedback setting, price rule and time based features.
            X_other_test = sub_test_set[dummy_vars_columns+time_columns].values
            X_all_test = np.concatenate([X_test_norm, X_other_test], axis=1)

            # predict allocative efficiency
            prediction = best_model.predict(X_all_test)[:, np.newaxis]            
            prediction = np.clip(prediction, a_min=0, a_max=1.0)

            # get the test targets
            test_targets = sub_test_set['allocative_efficiency_round'].values[:, np.newaxis]

            # Since allocative efficiency can be 0, calculate median APE, with some changes in the denominator.
            result_test_df = sub_test_set[key_columns].copy()
            denom = test_targets.copy()
            denom[denom==0]= prediction[denom==0]
            denom[denom==0] = 1

            # append to resutls
            result_test_df.loc[:, 'ae_ape'] = (np.abs(prediction - test_targets)/denom)
            result_test_df.loc[:, 'sample_id'] = i
            all_results.append(result_test_df)


  0%|          | 0/50 [00:00<?, ?it/s]

## Combine Model Results

In [14]:
regression_data_df = pd.concat(regression_res, axis=0, ignore_index = True)
all_results_df = pd.concat(all_results, ignore_index = True)
all_results_df['model'] = model_name

## Persist results to files

In [15]:
regression_data_df.to_feather('../../../data/results/allocative_efficiency/'+model_name+'_data.ft')
all_results_df.to_feather('../../../data/results/allocative_efficiency/'+model_name+'.ft')